# Market Regime Forecasting

## Objective

The objective of this notebook is to develop and evaluate machine learning models capable of forecasting future financial market regimes using historical regime information and engineered financial features. Rather than predicting future asset prices, the focus is on anticipating changes in market behavior by learning the temporal relationships between previously identified market regimes.

Building upon the statistical characterization and dynamic analysis performed in the previous notebooks, this stage investigates whether market regimes exhibit sufficient temporal structure to enable reliable forecasting. The resulting models establish the predictive component of the market regime detection framework and provide the foundation for strategy evaluation in the subsequent notebook.

## Roadmap

This notebook is organized into the following stages:

1. Environment Setup
2. Historical Forecasting Dataset Preparation
3. Encoding Categorical Variables
4. Feature Matrix Construction
5. Train-Test Split
6. Model Training
7. Model Evaluation

---

## Environment Setup

Before beginning the market regime dynamics analysis, the required libraries, project configuration, and datasets are loaded. This initialization step ensures that all subsequent analyses are performed using a consistent and reproducible environment.

In [1]:
# ==========================================
# Import Libraries and Load Configuration
# ==========================================
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap
import numpy as np
import seaborn as sns
import os 
import sys

# Machine Learning Libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Absoulte Path To The Project Route
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

from config import *

In [2]:
# ==========================================
# Asset Names
# ==========================================
if len(ASSETS) != 1:
    raise ValueError(
        "This notebook is designed for single-asset analysis."
    )

asset_name = (
    ASSETS[0]
    .lower()
    .replace("-", "_")
)

In [3]:
# ==========================================
# Load Dataset
# ==========================================
market_regime_path = os.path.join(
    PROCESSED_DATA_PATH,
    f"{asset_name}_market_regime.csv"
)

market_regime_df = pd.read_csv(
    market_regime_path,
    parse_dates=["Date"],
    index_col = "Date"
)

In [4]:
# ==========================================
# Verify Dataset
# ==========================================
print("Dataset shape:", market_regime_df.shape)

market_regime_df.head()
market_regime_df.info()

Dataset shape: (3065, 5)
<class 'pandas.DataFrame'>
DatetimeIndex: 3065 entries, 2018-01-31 to 2026-06-22
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Returns        3065 non-null   float64
 1   Volatility     3065 non-null   float64
 2   Momentum       3065 non-null   float64
 3   Cluster_Id     3065 non-null   int64  
 4   Market_Regime  3065 non-null   str    
dtypes: float64(3), int64(1), str(1)
memory usage: 143.7 KB


----

## Historical Forecasting Dataset Preparation

Before a predictive model can be trained, the market regime dataset must be transformed into a supervised learning problem. While the previous notebooks focused on describing historical market behavior, forecasting requires defining both the explanatory variables (features) and the target variable that the model will learn to predict.

In this section, the historical market regime dataset is prepared for forecasting by creating the prediction target, selecting the variables that describe current market conditions, and organizing the data into a structure suitable for machine learning algorithms.

The objective is to ensure that every observation contains the information available at a given point in time while using the following market regime as the prediction target.

In [5]:
# ==========================================
# Create Working Dataset
# ==========================================
forecast_df = market_regime_df.copy()

In [6]:
# ==========================================
# Create Prediction Target
# ==========================================

# Create the target varaible by shifting the market regime
forecast_df["Next_Market_Regime"] = forecast_df["Market_Regime"].shift(-1)

forecast_df = forecast_df.dropna(subset=["Next_Market_Regime"])

forecast_df[["Market_Regime", "Next_Market_Regime"]].head(10)

,Market_Regime,Next_Market_Regime
Date,,
2018-01-31,High-Volatility Transition,High-Volatility Bearish
2018-02-01,High-Volatility Bearish,High-Volatility Bearish
2018-02-02,High-Volatility Bearish,High-Volatility Transition
2018-02-03,High-Volatility Transition,High-Volatility Bearish
2018-02-04,High-Volatility Bearish,High-Volatility Bearish
2018-02-05,High-Volatility Bearish,High-Volatility Transition
2018-02-06,High-Volatility Transition,High-Volatility Transition
2018-02-07,High-Volatility Transition,High-Volatility Transition
2018-02-08,High-Volatility Transition,High-Volatility Transition


The resulting dataset now represents a supervised learning problem. Each observation contains the market information available at a given point in time together with the market regime observed in the subsequent period. This target variable will be used throughout the remainder of the notebook to train and evaluate forecasting models capable of predicting future market regimes.

----

## Encoding Categorical Variables
The forecasting dataset currently contains the variable Market_Regime, which is represented by descriptive labels such as Low-Volatility Neutral and High-Volatility Bearish. While these names are meaningful for interpretation, most machine learning algorithms require numerical inputs and cannot operate directly on text-based categories.

In this section, the categorical market regimes are transformed into numerical representations through label encoding. This process assigns a unique integer to each market regime while preserving the distinction between categories. The same encoding will later be applied to the prediction target to ensure consistency during model training and evaluation.

In [7]:
# ==========================================
# Encode Market Regimes
# ==========================================
from sklearn.preprocessing import LabelEncoder

# Create independent encoders 
regime_encoder = LabelEncoder()
target_encoder = LabelEncoder()

# Encode predcitor 
forecast_df["Market_Regime_Encoded"] = regime_encoder.fit_transform(forecast_df["Market_Regime"])

# Encode prediction target
forecast_df["Next_Market_Regime_Encoded"] = target_encoder.fit_transform(forecast_df["Next_Market_Regime"])

# Display the encoding mapping 
encoding_mapping = pd.DataFrame({
    "Market_Regime": regime_encoder.classes_,
    "Encoded Value": range(len(regime_encoder.classes_))
})

encoding_mapping

,Market_Regime,Encoded Value
0,High-Volatility Bearish,0
1,High-Volatility Transition,1
2,Low-Volatility Neutral,2
3,Moderate-Volatility Bullish,3


The categorical market regime labels have been successfully transformed into numerical representations suitable for machine learning algorithms. The encoding preserves the correspondence between each market regime and its numerical identifier, allowing future model predictions to be translated back into interpretable financial market states.

-----

## Feature Matrix Construction

Before training a forecasting model, it is necessary to define which variables will be used as predictors and which variable will serve as the prediction target.

Unlike the clustering stage, where all engineered features were analyzed simultaneously to discover hidden market regimes, supervised learning requires an explicit separation between the explanatory variables (features) and the response variable (target).

In this project, the forecasting model will use the information available at the current time step—including the engineered financial features and the identified market regime—to predict the market regime observed in the following period. Selecting these variables establishes the input space for the machine learning algorithms developed in the remainder of this notebook.

In [8]:
# ==========================================
# Select Features and Target
# ==========================================

# Predictor variables
FEATURE_COLUMNS = [
    "Returns",
    "Volatility",
    "Momentum",
    "Market_Regime_Encoded"
]

# Predictor target
TARGET_COLUMN = "Next_Market_Regime_Encoded"

# Separate predictors and target
x = forecast_df[FEATURE_COLUMNS].copy()
y = forecast_df[TARGET_COLUMN].copy()

# Display dimensions
print(f"Feature Matrix Shape: {x.shape}")
print(f"Target Vector Shape: {y.shape}")

# Preview predictors
x.head()

Feature Matrix Shape: (3064, 4)
Target Vector Shape: (3064,)


,Returns,Volatility,Momentum,Market_Regime_Encoded
Date,,,,
2018-01-31,0.011359,0.064866,-0.086472,1
2018-02-01,-0.102783,0.064014,-0.200817,0
2018-02-02,-0.037052,0.063907,-0.239214,0
2018-02-03,0.038973,0.064239,-0.288723,1
2018-02-04,-0.097865,0.060821,-0.286471,0


The predictor variables and prediction target have now been defined. The feature matrix contains the information available at each observation, while the target variable represents the market regime observed in the subsequent period. This separation establishes the supervised learning dataset that will be used throughout the remainder of the forecasting pipeline.

-----

## Train-Test Split

Before training a forecasting model, the prepared dataset must be divided into separate training and testing subsets.

The training dataset is used to learn the relationships between the current market conditions and the subsequent market regime, while the testing dataset provides an independent evaluation of the model's predictive performance on previously unseen observations.

Because financial data follows a chronological order, the split must preserve the temporal structure of the dataset. Randomly shuffling observations would introduce information from the future into the training process, resulting in unrealistic performance estimates. Therefore, the data is divided chronologically so that the model is always evaluated on observations that occur after those used for training.

In [9]:
# ==========================================
# Chronological Train-Test Split
# ==========================================

# Split the dataset while preserving chronological order 
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20, shuffle=False)

# Dislay dataset dimensions
print("=" * 50)
print("Training Set")
print("=" * 50)
print(f"Features: {x_train.shape}")
print(f"Target:   {y_train.shape}")

print("\n" + "=" * 50)
print("Testing Set")
print("=" * 50)
print(f"Features: {x_test.shape}")
print(f"Target:   {y_test.shape}")

Training Set
Features: (2451, 4)
Target:   (2451,)

Testing Set
Features: (613, 4)
Target:   (613,)


The forecasting dataset has been successfully divided into chronological training and testing subsets. By preserving the temporal order of the observations, the evaluation process reflects a realistic forecasting scenario in which the model learns from historical market behavior and is assessed using future observations that were not available during training.

-----

## Model Training

With the forecasting dataset fully prepared, the next step is to train a machine learning model capable of predicting the next market regime based on the current market conditions.

The objective of this section is not only to generate predictions, but also to establish a baseline forecasting model that can be evaluated and compared with more sophisticated approaches in later stages of the project.

As an initial benchmark, a Random Forest Classifier is selected due to its robustness, ability to model non-linear relationships, and strong performance on structured tabular datasets. The model will learn the relationship between the engineered financial features, the current market regime, and the market regime observed in the following period.

The trained model will serve as the foundation for evaluating forecasting performance and understanding how predictable market regime dynamics are.

In [10]:
# ==========================================
# Train Random Forest Classifier
# ==========================================

from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest classifier
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
random_forest_model.fit(x_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

The Random Forest classifier has been successfully trained using the historical market observations contained in the training dataset. During this process, the model learned the relationships between the engineered financial features, the current market regime, and the market regime observed in the subsequent period.

At this stage, the model has not yet been evaluated. The next section focuses on assessing its predictive performance using previously unseen observations from the testing dataset to determine how effectively it can forecast future market regimes.

-----

## Model Evaluation

Training a machine learning model is only the first step in the forecasting process. To determine whether the model has learned meaningful relationships between current market conditions and future market regimes, its performance must be evaluated using previously unseen observations.

This section assesses the predictive capabilities of the Random Forest classifier using the testing dataset. Several evaluation metrics are employed to measure forecasting performance and identify potential strengths and weaknesses of the model.

In addition to overall accuracy, a detailed analysis of class-level performance and prediction errors will be conducted to better understand how effectively different market regimes can be forecasted.

>### Generate Predictions

Once the Random Forest classifier has been trained, the next step is to generate predictions using the testing dataset.

These predictions represent the market regimes forecasted by the model based solely on information that was available during training. They will serve as the foundation for all subsequent evaluation metrics, including overall accuaracy, class-level performance, confusion matrices, and feature importance analysis.

Comparing the predicted market regimes with the actual observed regimes allows the forecasting performance of the model to be assessed objectively.

In [11]:
# ==========================================
# Generate Predictions
# ==========================================

# Predict market regimes for the testing dataset
y_pred = random_forest_model.predict(x_test)

# Compare the first predictions with the actual values
prediction_results = pd.DataFrame({
    "Actual Regime": y_test.values,
    "Predicted Regime": y_pred
})

prediction_results.head(10)

,Actual Regime,Predicted Regime
0,2,2
1,2,2
2,2,2
3,2,2
4,2,2
5,2,2
6,3,3
7,2,2
8,2,2
9,2,2


Predictions have been successfully generated for all observations in the testing dataset. These predicted market regimes constitute the basis for evaluating the forecasting performance of the Random Forest classifier. The following sections compare the predicted and observed market regimes using several quantitative performance metrics.


> ### Accuracy Score
The first metric used to evaluate the forecasting model is the accuracy score.

Accuracy measures the proportion of correctly predicted market regimes relative to the total number of observations in the testing dataset. It provides an overall assessment of the model's predictive performance and serves as an initial benchmark before conducting a more detailed analysis of each individual market regime.

Although accuracy offers a useful summary of model performance, it does not reveal how well each market regime is predicted. Therefore, additional evaluation metrics will be examined in the following sections to obtain a more comprehensive understanding of the forecasting model.

In [12]:
# ==========================================
# Calculate Accuracy Score
# ==========================================

from sklearn.metrics import accuracy_score

# Compute model accuracy 
accuracy = accuracy_score(y_test, y_pred)

# Display the results
print(f"Model Accuracy: {accuracy:.4f}")
print(f"Model Accuracy: {accuracy:.2%}")

Model Accuracy: 0.8450
Model Accuracy: 84.50%


#### Accuracy Score Interpretation
The Random Forest classifier achieved an overall forecasting accuracy of **84.50%** on the testing dataset, indicating that the model correctly predicted the subsequent market regime for the majority of previously unseen observations.

This result suggests that the engineered financial features and the current market regime contain substantial predictive information regarding the short-term evolution of market conditions. Rather than relying on random chance, the model appears to have learned meaningful relationships between historical market behavior and future regime transitions.

Although an overall accuracy of 84.50% represents a strong initial forecasting performance, this metric alone does not provide a complete assessment of the model's predictive capabilities. In particular, accuracy does not reveal whether all market regimes are predicted equally well or whether certain regimes are systematically confused with others. Consequently, a more detailed evaluation using class-specific performance metrics and confusion matrices is required before drawing conclusions about the practical effectiveness of the forecasting model.